# Predicting Air Quality Index - High Score Time-Aware Ensemble

This notebook is built for competition platform execution:
- Input: `dataset/public/train.csv`, `dataset/public/test.csv`, `dataset/public/sample_submission.csv`
- Output: `working/submission.csv`
- Strict chronological validation (no random split leakage)
- Leakage-safe target encoding
- Multi-model ensemble with OOF optimization


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer

try:
    from lightgbm import LGBMRegressor
except Exception as e:
    raise ImportError('This notebook requires lightgbm. Please ensure lightgbm is available on the platform.') from e

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def locate_data_dir():
    candidates = [
        'dataset/public',
        './dataset/public',
        '/kaggle/input/predicting-air-quality-index',
        '/kaggle/input/predicting-air-quality-index-dataset',
        '/Users/songling/Desktop/Predicting Air Quality Index',
    ]
    for d in candidates:
        if all(os.path.exists(os.path.join(d, f)) for f in ['train.csv', 'test.csv', 'sample_submission.csv']):
            return d
    raise FileNotFoundError('Could not find train/test/sample_submission in known input paths.')


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


In [ ]:
DATA_DIR = locate_data_dir()
print('Using DATA_DIR:', DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print('train:', train.shape, 'test:', test.shape, 'sample:', sample_sub.shape)


In [ ]:
def add_features(df):
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce')

    # Time ordinal
    out['date_ordinal'] = out['date'].map(lambda x: x.toordinal() if pd.notna(x) else np.nan)
    out['weekofyear'] = out['date'].dt.isocalendar().week.astype('float64')
    out['dayofyear'] = out['date'].dt.dayofyear.astype('float64')

    # Cyclical transforms
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24.0)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24.0)
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12.0)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12.0)
    out['dow_sin'] = np.sin(2 * np.pi * (out['day_of_week'].map({
        'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6
    }).fillna(0)) / 7.0)
    out['dow_cos'] = np.cos(2 * np.pi * (out['day_of_week'].map({
        'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6
    }).fillna(0)) / 7.0)
    out['doy_sin'] = np.sin(2 * np.pi * out['dayofyear'] / 365.25)
    out['doy_cos'] = np.cos(2 * np.pi * out['dayofyear'] / 365.25)

    # Weather interactions
    out['temp_humidity'] = out['temperature'] * out['humidity']
    out['temp_wind'] = out['temperature'] * out['wind_speed']
    out['hum_wind'] = out['humidity'] * out['wind_speed']
    out['wind_vis_ratio'] = out['wind_speed'] / (out['visibility'] + 1e-3)
    out['temp_hum_ratio'] = out['temperature'] / (out['humidity'] + 1e-3)
    out['visibility_inv'] = 1.0 / (out['visibility'] + 1e-3)
    out['is_rush_hour'] = out['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    out['is_night'] = out['hour'].isin([0,1,2,3,4,5,22,23]).astype(int)

    # Useful category crosses
    out['city_station'] = out['city'].astype(str) + '_' + out['station'].astype(str)
    out['station_hour'] = out['station'].astype(str) + '_' + out['hour'].astype(str)
    out['station_month'] = out['station'].astype(str) + '_' + out['month'].astype(str)
    out['city_season'] = out['city'].astype(str) + '_' + out['season'].astype(str)

    out = out.drop(columns=['date'])
    return out


def smooth_target_encode(ref_df, ref_y, apply_df, col, m=100):
    global_mean = float(ref_y.mean())
    tmp = ref_df[[col]].copy()
    tmp['_y'] = ref_y.values
    grp = tmp.groupby(col)['_y'].agg(['mean', 'count'])
    enc = (grp['mean'] * grp['count'] + global_mean * m) / (grp['count'] + m)
    return apply_df[col].map(enc).fillna(global_mean)


def build_lgb_matrices(train_base, apply_base, y_train):
    tr = train_base.copy()
    ap = apply_base.copy()

    te_cols = [
        'city', 'station', 'season', 'day_of_week',
        'hour', 'month', 'city_station', 'station_hour', 'station_month', 'city_season'
    ]
    for c in te_cols:
        tr[f'te_{c}'] = smooth_target_encode(tr, y_train, tr, c, m=120)
        ap[f'te_{c}'] = smooth_target_encode(tr, y_train, ap, c, m=120)

    cat_cols = ['day_of_week', 'season', 'city', 'station', 'city_station', 'station_hour', 'station_month', 'city_season']
    for c in cat_cols:
        tr[c] = tr[c].astype('category')
        ap[c] = ap[c].astype('category')

    return tr, ap


In [ ]:
train_fe = add_features(train)
test_fe = add_features(test)

y = train_fe['aqi'].astype(float)
X_base = train_fe.drop(columns=['aqi', 'id']).copy()
X_test_base = test_fe.drop(columns=['id']).copy()

# Chronological folds by date order (future-aware)
dates = pd.to_datetime(train['date'], errors='coerce')
order = np.argsort(dates.values)
folds = 5
chunks = np.array_split(order, folds + 1)  # expanding-window style

split_indices = []
for i in range(1, len(chunks)):
    va_idx = chunks[i]
    tr_idx = np.concatenate(chunks[:i])
    if len(tr_idx) == 0 or len(va_idx) == 0:
        continue
    split_indices.append((tr_idx, va_idx))

print('Number of chronological folds:', len(split_indices))
for i, (tr_idx, va_idx) in enumerate(split_indices, 1):
    print(f'Fold {i}: train={len(tr_idx)} valid={len(va_idx)}')


In [ ]:
# Base model configs
lgb_configs = [
    {
        'name': 'lgb_a',
        'params': dict(
            objective='regression',
            metric='l2',
            n_estimators=2600,
            learning_rate=0.018,
            num_leaves=192,
            min_child_samples=20,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_alpha=0.15,
            reg_lambda=1.4,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    },
    {
        'name': 'lgb_b',
        'params': dict(
            objective='regression',
            metric='l2',
            boosting_type='goss',
            n_estimators=2200,
            learning_rate=0.020,
            num_leaves=128,
            min_child_samples=25,
            subsample=1.0,
            colsample_bytree=0.85,
            reg_alpha=0.10,
            reg_lambda=1.8,
            random_state=RANDOM_STATE + 11,
            n_jobs=-1
        )
    },
    {
        'name': 'lgb_c',
        'params': dict(
            objective='regression',
            metric='l2',
            n_estimators=3000,
            learning_rate=0.015,
            num_leaves=224,
            min_child_samples=16,
            subsample=0.95,
            colsample_bytree=0.95,
            reg_alpha=0.05,
            reg_lambda=1.2,
            random_state=RANDOM_STATE + 23,
            n_jobs=-1
        )
    }
]

cat_cols_et = ['day_of_week', 'season', 'city', 'station', 'city_station', 'station_hour', 'station_month', 'city_season']
num_cols_et = [c for c in X_base.columns if c not in cat_cols_et]

etr_pre = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols_et),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', make_ohe())
        ]), cat_cols_et)
    ],
    remainder='drop'
)

etr_model_template = TransformedTargetRegressor(
    regressor=Pipeline([
        ('prep', etr_pre),
        ('reg', ExtraTreesRegressor(
            n_estimators=500,
            min_samples_leaf=2,
            max_features=0.85,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)


In [ ]:
# OOF training for robust ensemble weighting
oof_preds = {}
test_preds = {}

# LightGBM family
for cfg in lgb_configs:
    name = cfg['name']
    oof = np.zeros(len(X_base), dtype=float)
    test_fold_preds = []

    for fold_id, (tr_idx, va_idx) in enumerate(split_indices, 1):
        X_tr_base = X_base.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        X_va_base = X_base.iloc[va_idx].copy()

        X_tr_lgb, X_va_lgb = build_lgb_matrices(X_tr_base, X_va_base, y_tr)
        model = LGBMRegressor(**cfg['params'])
        model.fit(X_tr_lgb, y_tr)

        oof[va_idx] = model.predict(X_va_lgb)

        # fold test prediction uses the same fold train statistics (safe)
        _, X_te_lgb = build_lgb_matrices(X_tr_base, X_test_base, y_tr)
        test_fold_preds.append(model.predict(X_te_lgb))

    oof_preds[name] = oof
    test_preds[name] = np.mean(np.vstack(test_fold_preds), axis=0)

# ExtraTrees family
oof_etr = np.zeros(len(X_base), dtype=float)
test_fold_preds_etr = []

for fold_id, (tr_idx, va_idx) in enumerate(split_indices, 1):
    X_tr = X_base.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx].copy()
    X_va = X_base.iloc[va_idx].copy()

    model = etr_model_template
    model.fit(X_tr, y_tr)
    oof_etr[va_idx] = model.predict(X_va)
    test_fold_preds_etr.append(model.predict(X_test_base))

oof_preds['etr'] = oof_etr
test_preds['etr'] = np.mean(np.vstack(test_fold_preds_etr), axis=0)

# Show each model OOF MSE (lower is better)
for k, p in oof_preds.items():
    mask = p != 0
    mse = mean_squared_error(y[mask], p[mask])
    print(f'{k} OOF MSE: {mse:.6f}')


In [ ]:
# Optimize non-negative blend weights on OOF predictions
model_names = list(oof_preds.keys())
P = np.column_stack([oof_preds[m] for m in model_names])

# Use rows that have OOF predictions (all validation chunks)
valid_rows = np.any(P != 0, axis=1)
Pv = P[valid_rows]
yv = y.values[valid_rows]

best_mse = 1e18
best_w = None
rng = np.random.default_rng(RANDOM_STATE)

# random search on simplex
for _ in range(80000):
    w = rng.dirichlet(np.ones(len(model_names)) * 1.25)
    pred = Pv @ w
    mse = mean_squared_error(yv, pred)
    if mse < best_mse:
        best_mse = mse
        best_w = w

print('Best OOF blended MSE:', best_mse)
print('Blend weights:', {m: float(w) for m, w in zip(model_names, best_w)})


In [ ]:
# Final submission
final_test_pred = np.zeros(len(X_test_base), dtype=float)
for i, m in enumerate(model_names):
    final_test_pred += best_w[i] * test_preds[m]

final_test_pred = np.clip(final_test_pred, 25.0, 500.0)

submission = pd.DataFrame({
    'id': test['id'].astype(int),
    'aqi': final_test_pred
}).sort_values('id').reset_index(drop=True)

assert len(submission) == len(test), 'Submission row count mismatch.'

os.makedirs('working', exist_ok=True)
out_path = 'working/submission.csv'
submission.to_csv(out_path, index=False)

print('Saved:', out_path)
print('Shape:', submission.shape)
print(submission.head())
